GOLD LAYER — CUSTOMER PERFORMANCE

Table: gold.customer_performance

Grain: 1 row = 1 customer

Business goal: Analyze customer value and behavior:

Total revenue per customer
Customer purchase frequency
Customer lifetime (first vs last order)
Average order value
Loyalty tier contribution

1. Load Source Tables

In [0]:
df_sales = spark.table("silver.sales_orders")
df_customers = spark.table("silver.crm_customers")

2. Aggregate Customer Metrics

In [0]:
from pyspark.sql.functions import sum, countDistinct, min, max, round

df_customer_metrics = df_sales.groupBy("customer_id").agg(
    sum("total_amount").alias("total_revenue"),
    countDistinct("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity"),
    round(sum("total_amount") / countDistinct("order_id"), 2).alias("avg_order_value"),
    min("order_date").alias("first_order_date"),
    max("order_date").alias("last_order_date")
)

3. Join Customer Attributes (CRM)

In [0]:
df_customer_perf = df_customer_metrics.join(
    df_customers,
    on="customer_id",
    how="left"
)

In [0]:
from pyspark.sql.functions import datediff

df_customer_perf = df_customer_perf.withColumn(
    "customer_lifetime_days",
    datediff("last_order_date", "first_order_date")
)

4. Select Final Columns (Clean Schema)

In [0]:
df_customer_perf = df_customer_perf.select(
    "customer_id",
    "customer_name",
    "loyalty_tier",
    "total_revenue",
    "total_orders",
    "total_quantity",
    "avg_order_value",
    "first_order_date",
    "last_order_date",
    "customer_lifetime_days"   
)

5. Write to Gold Table

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("DROP TABLE IF EXISTS gold.customer_performance")

df_customer_perf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.customer_performance")

6. Validation

In [0]:
display(spark.table("gold.customer_performance"))